# Belgian Urban Heat Monitor
### Hyperlocal Urban Heat Anomaly Detection · Leuven + Belgium · 2023–2025

> **AI-assisted urban heat anomaly prediction using satellite imagery, citizen science sensors, and official meteorological reference data.**

This pipeline detects **where** heat concentrates within cities — not just how hot it is.  
It compares Leuven (dense hyperlocal coverage) with other Belgian cities using the RMI official station network,  
and is designed to power a **Belgian Urban Heat Dashboard** website.

| Data source | Coverage | Role |
|---|---|---|
| **Leuven.cool** (Zenodo) | ~110 stations · Leuven | Hyperlocal temperature variation |
| **RMI AWS** (opendata.meteo.be) | 17+ stations · Belgium-wide | Official baseline temperature reference |
| **Sentinel-2 L2A** (Copernicus) | 7 scenes · 2023–2025 | Spatial context: NDVI, NDBI |

---

## Table of Contents
1. [Setup & Config](#1-setup--config)
2. [Phase 1 — Data Collection](#2-phase-1--data-collection)
3. [Phase 2 — Preprocessing & EDA](#3-phase-2--preprocessing--eda)
4. [Phase 3 — Heat Anomaly Target Engineering](#4-phase-3--heat-anomaly-target-engineering)
5. [Phase 4 — Sentinel-2 Feature Engineering](#5-phase-4--sentinel-2-feature-engineering)
6. [Phase 5 — ML Modeling](#6-phase-5--ml-modeling)
7. [Phase 6 — Evaluation & Model Comparison](#7-phase-6--evaluation--model-comparison)
8. [Phase 7 — Heat Risk Maps](#8-phase-7--heat-risk-maps)
9. [Phase 8 — Belgian Urban Heat Dashboard Vision](#9-phase-8--belgian-urban-heat-dashboard-vision)
10. [Phase 9 — Deployment](#10-phase-9--deployment)

---
## 1. Setup & Config

In [12]:
import sys
print(sys.executable)

import sys
!"{sys.executable}" -m pip show openkmi

import openkmi

print(openkmi)
print(dir(openkmi))



c:\Users\wamy\Desktop\urban-heat-ai\.venv\Scripts\python.exe
Name: openkmi
Version: 0.7.0
Summary: Python package to download open data from KMI
Home-page: https://github.com/TimFranken/openkmi
Author: Tim Franken
Author-email: tim.franken@sumaqua.be
License: mit
Location: c:\Users\wamy\Desktop\urban-heat-ai\.venv\Lib\site-packages
Requires: owslib, pandas
Required-by: 
<module 'openkmi' from 'c:\\Users\\wamy\\Desktop\\urban-heat-ai\\.venv\\Lib\\site-packages\\openkmi\\__init__.py'>
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [11]:
import warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from pathlib import Path
warnings.filterwarnings('ignore')

# ML
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (KFold, cross_val_score,
                                     RandomizedSearchCV, StratifiedShuffleSplit)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

# Deep Learning — CNN
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split as tts

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns

# RMI open data
try:
    from openkmi.stations import Stations
    from openkmi.observations import Observations
    HAS_OPENKMI = True
    print('openkmi available')
except ImportError:
    HAS_OPENKMI = False
    print('openkmi not installed — install with: pip install openkmi')
    print('RMI cells will run in stub mode')

print('All imports OK')

openkmi not installed — install with: pip install openkmi
RMI cells will run in stub mode
All imports OK


In [3]:
# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..')
DATA_ROOT    = PROJECT_ROOT / 'data' / 'raw'
LEUVEN_DIR   = DATA_ROOT / 'leuven'
SENTINEL_DIR = DATA_ROOT / 'sentinel2'
STATIONS_CSV = LEUVEN_DIR / 'STATIONS.csv'

# ── Sentinel-2 scene inventory (7 low-cloud scenes from Copernicus) ────────────
SCENE_DATES = pd.to_datetime([
    '2023-08-10', '2023-08-20', '2023-08-23',
    '2024-07-30',
    '2025-07-02', '2025-08-11', '2025-08-12',
])

# Representative mid-summer scene per year for maps
REPR_SCENES = {2023: '2023-08-20', 2024: '2024-07-30', 2025: '2025-08-11'}

# ── Date filter: keep sensor rows within ±N days of any scene ─────────────────
SCENE_WINDOW_DAYS = 7   # sensor data must be within 7 days of a scene

# ── Other settings ─────────────────────────────────────────────────────────────
PATCH_SIZE   = 16
N_FOLDS      = 5
RANDOM_STATE = 42
YEARS        = [2023, 2024, 2025]
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

# Leuven bounding box
LAT_MIN, LAT_MAX = 50.84, 50.94
LON_MIN, LON_MAX = 4.63,  4.78

print(f'Device        : {DEVICE}')
print(f'Scene window  : ±{SCENE_WINDOW_DAYS} days')
print(f'Scenes        : {len(SCENE_DATES)}')

Device        : cpu
Scene window  : ±7 days
Scenes        : 7


---
## 2. Phase 1 — Data Collection

### 2a. Data Quality Statement

> ⚠️ **Known limitations — documented transparently:**
>
> | Issue | Impact | Mitigation |
> |---|---|---|
> | Only **1 Sentinel-2 scene** for 2024 (Jul 30) | 2024 RMSE ~2× worse than 2023 | Acknowledged; 2024 excluded from spatial maps |
> | Previous `nrows=50_000` cap loaded mainly July data | Scene dates are Aug → temporal mismatch → only 266 merged rows | **Fixed:** filter by ±7-day window around each scene |
> | CNN trained on 266 rows → R² = −0.23 | Underfitting, expected | **Fixed:** more rows after window filter; CNN as demonstration |
> | Predicted range was only 1°C (19.7–20.7°C) | Model learned absolute temperature, not spatial pattern | **Fixed:** target is now `temp_anomaly` (°C above city mean) |

### 2b. Leuven.cool Stations

In [4]:
stations = pd.read_csv(STATIONS_CSV)
stations.columns = stations.columns.str.upper().str.strip()
print(f'{len(stations)} Leuven.cool stations loaded')
stations.head()

155 Leuven.cool stations loaded


,ID,WOWID,LATITUDE,LONGITUDE,ALTITUDE
0,GARMON01,b1f9850c-d686-e911-80e7-0003ff59889d,50.8710,4.694,21
1,GARMON002,83a89aa6-2695-e911-80e7-0003ff59883f,50.8468,4.756,47
2,GARMON003,2a3596b2-2795-e911-80e7-0003ff59889d,50.8700,4.728,44
3,GARMON004,7d43d8ab-2895-e911-80e7-0003ff59883f,50.8708,4.685,31
4,GARMON005,3f67b310-5597-e911-80e7-0003ff59883f,50.8814,4.713,26


In [5]:
# ── Visual 1: Station network map ─────────────────────────────────────────────
fig_stn = px.scatter_mapbox(
    stations, lat='LATITUDE', lon='LONGITUDE',
    hover_name='ID',
    hover_data={'ALTITUDE': True, 'LATITUDE': ':.4f', 'LONGITUDE': ':.4f'},
    color='ALTITUDE', color_continuous_scale='Viridis',
    zoom=12, mapbox_style='carto-positron',
    title='Leuven.cool Sensor Network — 155 stations coloured by altitude',
    height=480
)
fig_stn.show()

### 2c. Leuven.cool Sensor Data — Scene-Windowed Loading

**Key fix:** instead of `nrows=50_000` (which loads mostly July data while scenes are in August),  
we load the full Q3 file and **keep only rows within ±7 days of a Sentinel-2 scene date**.  
This ensures sensor data and imagery are temporally matched.

In [6]:
SENSOR_COLS = [
    'ID', 'TEMPF', 'HUMIDITY', 'DEWPTF',
    'SOLARRADIATION', 'WINDSPEEDMPH', 'WINDGUSTMPH',
    'WINDDIR', 'RAININ', 'DATEUTC'
]

# Build date ranges: [scene - window, scene + window] for each scene
scene_windows = pd.IntervalIndex.from_arrays(
    SCENE_DATES - pd.Timedelta(days=SCENE_WINDOW_DAYS),
    SCENE_DATES + pd.Timedelta(days=SCENE_WINDOW_DAYS),
    closed='both'
)

def is_near_scene(dt_series):
    """Return boolean mask: True if date is within ±7 days of any scene."""
    dt_tz_naive = dt_series.dt.tz_localize(None)
    mask = pd.Series(False, index=dt_series.index)
    for scene in SCENE_DATES:
        lo = scene - pd.Timedelta(days=SCENE_WINDOW_DAYS)
        hi = scene + pd.Timedelta(days=SCENE_WINDOW_DAYS)
        mask |= dt_tz_naive.between(lo, hi)
    return mask


def load_sensor_year(year):
    """Load full Q3 file, then filter to scene windows only."""
    path = LEUVEN_DIR / f'RAWDATA{year}Q3.csv'
    chunks, kept = [], 0

    # Read in chunks to avoid loading the full file into RAM
    for chunk in pd.read_csv(
        path,
        usecols=lambda c: c.upper() in SENSOR_COLS,
        chunksize=200_000,
        na_values=['NULL', '', 'NA']
    ):
        chunk.columns = chunk.columns.str.upper().str.strip()
        chunk['DATEUTC'] = pd.to_datetime(chunk['DATEUTC'], errors='coerce', utc=True)
        chunk = chunk[is_near_scene(chunk['DATEUTC'])]
        if len(chunk) > 0:
            chunks.append(chunk)
            kept += len(chunk)

    df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame()
    df['year'] = year
    print(f'  {year}: {kept:,} rows kept within scene windows')
    return df


print('Loading sensor data (scene-windowed)...')
raw_dfs = {yr: load_sensor_year(yr) for yr in YEARS}
raw_all = pd.concat(raw_dfs.values(), ignore_index=True)
print(f'Total: {len(raw_all):,} rows')
raw_all.head(3)

Loading sensor data (scene-windowed)...
  2023: 15,144,729 rows kept within scene windows
  2024: 7,107,196 rows kept within scene windows
  2025: 8,642,965 rows kept within scene windows
Total: 30,894,890 rows


,ID,TEMPF,HUMIDITY,DEWPTF,WINDDIR,WINDSPEEDMPH,WINDGUSTMPH,RAININ,SOLARRADIATION,DATEUTC,year
0,GARMON075,60.6,97,59.7,7,0.67,2.46,0.00,0.0,2023-08-03 00:00:00+00:00,2023
1,GARMONR01,59.7,99,59.5,74,2.46,2.46,0.07,0.0,2023-08-03 00:00:00+00:00,2023
2,GARMONR05,59.4,93,57.4,214,1.57,4.92,0.00,0.0,2023-08-03 00:00:02+00:00,2023


### 2d. RMI AWS — Official Belgium Meteorological Reference

The [Royal Meteorological Institute open data](https://opendata.meteo.be) provides **17+ official AWS stations** across Belgium  
with calibrated, quality-controlled 10-minute observations.  
These serve as the **baseline reference** — what was Belgium's actual temperature that day,  
independent of Leuven.cool's citizen science network.

Install: `pip install openkmi`

In [7]:
# RMI stations relevant to Leuven + Belgian cities
RMI_CITIES = {
    'Leuven (Diest)': 'IRM_DIEST',
    'Brussels':       'IRM_UCL',
    'Ghent':          'IRM_GHENT',
    'Antwerp':        'IRM_ANTWERPEN',
    'Liège':          'IRM_LIEGE',
}

if HAS_OPENKMI:
    obs  = Observations()
    stns = Stations()

    # Pull daily max temperature for summer 2023-2025
    rmi_records = []
    for city, station_id in RMI_CITIES.items():
        try:
            df_rmi = obs.get_data(
                station=station_id,
                start='2023-07-01',
                end='2025-09-30',
                parameter='TEMP_DRY_SHELTER_AVG'
            )
            df_rmi['city']       = city
            df_rmi['station_id'] = station_id
            rmi_records.append(df_rmi)
            print(f'  {city}: {len(df_rmi):,} rows')
        except Exception as e:
            print(f'  {city}: FAILED — {e}')

    rmi_df = pd.concat(rmi_records, ignore_index=True) if rmi_records else pd.DataFrame()

else:
    # ── Stub — realistic synthetic RMI data ───────────────────────────────────
    print('Running in stub mode — generating synthetic RMI reference data')
    rng = np.random.default_rng(RANDOM_STATE)
    dates = pd.date_range('2023-07-01', '2025-09-30', freq='D')
    records = []
    baselines = {
        'Leuven (Diest)': 17.8, 'Brussels': 18.2,
        'Ghent': 17.5, 'Antwerp': 18.0, 'Liège': 17.2
    }
    for city, base in baselines.items():
        # Seasonal signal + noise
        doy = pd.Series(dates.dayofyear)
        seasonal = base + 6 * np.sin((doy - 80) * 2 * np.pi / 365)
        noise    = rng.normal(0, 1.2, len(dates))
        for i, d in enumerate(dates):
            records.append({
                'datetime':  d,
                'value':     round(float(seasonal.iloc[i] + noise[i]), 2),
                'city':      city,
                'parameter': 'TEMP_DRY_SHELTER_AVG'
            })
    rmi_df = pd.DataFrame(records)

rmi_df['datetime'] = pd.to_datetime(rmi_df['datetime'])
rmi_df['date']     = rmi_df['datetime'].dt.normalize()
print(f'RMI data: {len(rmi_df):,} rows across {rmi_df["city"].nunique()} cities')

Running in stub mode — generating synthetic RMI reference data
RMI data: 4,115 rows across 5 cities


In [8]:
# ── Visual 2: RMI temperature timeseries — Belgian cities comparison ──────────
rmi_daily = rmi_df.groupby(['date','city'])['value'].mean().reset_index()
# Show only Q3 months
rmi_q3 = rmi_daily[rmi_daily['date'].dt.month.isin([7,8,9])]

fig_rmi = px.line(
    rmi_q3, x='date', y='value', color='city',
    labels={'value': 'Temperature (°C)', 'date': 'Date', 'city': 'City'},
    title='RMI Official Temperature — Belgian Cities · Summer Q3 2023–2025',
    template='plotly_white', height=420
)
fig_rmi.show()

In [9]:
# ── Visual 3: City temperature distributions — inter-city comparison ──────────
fig_city_box = px.box(
    rmi_q3, x='city', y='value', color='city',
    labels={'value': 'Temperature (°C)', 'city': 'City'},
    title='Summer Temperature Distribution by Belgian City (RMI AWS)',
    template='plotly_white', height=400
)
fig_city_box.update_layout(showlegend=False)
fig_city_box.show()

In [10]:
# ── Visual 4: Annual heat anomaly — year-over-year spike detection ─────────────
# Compute each year's summer mean vs 2023 baseline
rmi_q3_copy = rmi_q3.copy()
rmi_q3_copy['year'] = rmi_q3_copy['date'].dt.year
baseline = rmi_q3_copy[rmi_q3_copy['year'] == 2023].groupby('city')['value'].mean()

yearly_mean = rmi_q3_copy.groupby(['year','city'])['value'].mean().reset_index()
yearly_mean = yearly_mean.merge(baseline.rename('baseline'), on='city')
yearly_mean['anomaly_vs_2023'] = (yearly_mean['value'] - yearly_mean['baseline']).round(2)

fig_spike = px.bar(
    yearly_mean, x='city', y='anomaly_vs_2023', color='year',
    barmode='group',
    labels={'anomaly_vs_2023': 'Temperature anomaly vs 2023 (°C)', 'city': 'City'},
    title='Year-over-Year Heat Anomaly — Is 2024 or 2025 Hotter than 2023?',
    color_discrete_sequence=['#4C72B0','#DD8452','#55A868'],
    template='plotly_white', height=400
)
fig_spike.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig_spike.show()

---
## 3. Phase 2 — Preprocessing & EDA

### 3a. Sensor Cleaning

In [11]:
df = raw_all.copy()

# Unit conversions
df['temp_c']  = (df['TEMPF']  - 32) * 5/9
df['dewpt_c'] = (df['DEWPTF'] - 32) * 5/9
df['wind_ms'] = df['WINDSPEEDMPH'] * 0.44704
df['gust_ms'] = df['WINDGUSTMPH']  * 0.44704

# Drop nulls and physical implausibles
df = df.dropna(subset=['temp_c', 'HUMIDITY', 'SOLARRADIATION'])
df = df[
    df['temp_c'].between(-5, 50) &
    df['HUMIDITY'].between(0, 100) &
    (df['SOLARRADIATION'] >= 0)
]

# Merge station metadata
df = df.merge(stations[['ID','LATITUDE','LONGITUDE','ALTITUDE']], on='ID', how='left')

# Clip to Leuven AOI
df = df[
    df['LATITUDE'].between(LAT_MIN, LAT_MAX) &
    df['LONGITUDE'].between(LON_MIN, LON_MAX)
]

df['date'] = df['DATEUTC'].dt.tz_localize(None).dt.normalize()
df['hour'] = df['DATEUTC'].dt.hour

print(f'After cleaning: {len(df):,} rows | {df["ID"].nunique()} unique stations')
df[['temp_c','HUMIDITY','SOLARRADIATION','wind_ms','ALTITUDE']].describe().round(2)

After cleaning: 28,287,674 rows | 111 unique stations


,temp_c,HUMIDITY,SOLARRADIATION,wind_ms,ALTITUDE
count,28287674.00,28287674.00,28287674.00,28287674.00,28287674.00
mean,20.06,76.75,102.44,0.22,33.26
std,5.01,17.38,154.63,0.56,14.44
min,7.39,23.00,0.00,0.00,11.00
25%,16.50,64.00,0.00,0.00,21.00
50%,19.50,80.00,24.44,0.00,31.00
75%,23.22,92.00,138.89,0.00,42.00
max,41.61,99.00,1554.28,13.60,92.00


In [ ]:
# ── Visual 5: Temperature distribution by year ────────────────────────────────
fig_dist = px.histogram(
    df, x='temp_c', color='year',
    nbins=60, barmode='overlay', opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'temp_c': 'Temperature (°C)'},
    title='Temperature Distribution — Scene-windowed Q3 data by Year',
    template='plotly_white', height=380
)
fig_dist.show()

In [ ]:
# ── Visual 6: Temperature box-plot per year ───────────────────────────────────
fig_box = px.box(
    df, x='year', y='temp_c', color='year',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'temp_c': 'Temperature (°C)'},
    title='Temperature Spread per Year (scene-windowed, all stations)',
    template='plotly_white', height=380
)
fig_box.show()

In [ ]:
# ── Visual 7: Correlation heatmap ─────────────────────────────────────────────
corr_cols = ['temp_c','HUMIDITY','SOLARRADIATION','wind_ms','ALTITUDE','dewpt_c']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix', fontsize=13, pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# ── Visual 8: Diurnal temperature cycle per year ──────────────────────────────
diurnal = df.groupby(['year','hour'])['temp_c'].mean().reset_index()

fig_diurnal = px.line(
    diurnal, x='hour', y='temp_c', color='year',
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'hour':'Hour (UTC)', 'temp_c':'Mean Temperature (°C)'},
    title='Average Diurnal Temperature Cycle — scene-windowed Q3 by Year',
    template='plotly_white', height=380
)
fig_diurnal.show()

---
## 4. Phase 3 — Heat Anomaly Target Engineering

### Why `temp_anomaly` instead of `temp_c`?

| Problem with `temp_c` | Why it breaks spatial mapping |
|---|---|
| All Leuven stations share the same weather → small absolute range | NDVI/NDBI explain only ±0.5°C → everything looks the same colour |
| The model learns "it was August, so ~20°C" | Not "this park is 2°C cooler than the road next to it" |
| Grid predictions collapse to 1°C range | Urban heat island signal is invisible |

**Fix:** `temp_anomaly = temp_c − city_daily_mean`  
Now a park with NDVI=0.7 might have anomaly = −2°C and a parking lot with NDBI=0.4 might have +3°C.  
The model learns **spatial heat patterns from satellite features**, not the regional weather.

In [ ]:
# Daily mean per station
sensor_daily = (
    df.groupby(['ID', 'date', 'year'])
      .agg(
          temp_c         = ('temp_c',        'mean'),
          HUMIDITY       = ('HUMIDITY',       'mean'),
          SOLARRADIATION = ('SOLARRADIATION', 'mean'),
          wind_ms        = ('wind_ms',        'mean'),
          ALTITUDE       = ('ALTITUDE',       'first'),
          LATITUDE       = ('LATITUDE',       'first'),
          LONGITUDE      = ('LONGITUDE',      'first'),
      ).reset_index()
)

# City-wide daily mean temperature (across all stations)
city_daily_mean = (
    sensor_daily.groupby('date')['temp_c']
                .mean()
                .rename('city_mean_temp')
)

# Compute anomaly: how much hotter/cooler is this station vs city average?
sensor_daily = sensor_daily.merge(city_daily_mean, on='date')
sensor_daily['temp_anomaly'] = sensor_daily['temp_c'] - sensor_daily['city_mean_temp']

print(f'Anomaly range: {sensor_daily["temp_anomaly"].min():.2f} to {sensor_daily["temp_anomaly"].max():.2f} C')
print(f'Anomaly std  : {sensor_daily["temp_anomaly"].std():.3f} C')
sensor_daily[['temp_c','city_mean_temp','temp_anomaly']].describe().round(3)

In [ ]:
# ── Visual 9: Anomaly distribution — now showing real spatial variation ────────
fig_anom = px.histogram(
    sensor_daily, x='temp_anomaly', color='year',
    nbins=60, barmode='overlay', opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'temp_anomaly': 'Temperature Anomaly (°C vs city mean)'},
    title='Heat Anomaly Distribution — spatial variation is now visible',
    template='plotly_white', height=380
)
fig_anom.add_vline(x=0, line_dash='dash', line_color='gray',
                   annotation_text='City mean', annotation_position='top right')
fig_anom.show()

In [ ]:
# ── Visual 10: Station-level anomaly map — hotspots visible ───────────────────
stn_anomaly = (
    sensor_daily.groupby(['ID','LATITUDE','LONGITUDE','ALTITUDE','year'])['temp_anomaly']
                .mean().reset_index()
)

# Use overall mean across years for the map
stn_overall = (
    sensor_daily.groupby(['ID','LATITUDE','LONGITUDE','ALTITUDE'])['temp_anomaly']
                .mean().reset_index().rename(columns={'temp_anomaly':'mean_anomaly'})
)

fig_stn_anom = px.scatter_mapbox(
    stn_overall.dropna(subset=['LATITUDE','LONGITUDE']),
    lat='LATITUDE', lon='LONGITUDE',
    color='mean_anomaly',
    size=stn_overall['mean_anomaly'].abs() + 0.1,
    hover_name='ID',
    hover_data={'mean_anomaly': ':.2f', 'ALTITUDE': True},
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0,
    zoom=12, mapbox_style='carto-positron',
    title='Mean Temperature Anomaly per Station — Blue=Cool, Red=Hot',
    height=520
)
fig_stn_anom.show()

In [ ]:
# ── Visual 11: Altitude vs anomaly — do higher stations run cooler? ────────────
fig_alt_anom = px.scatter(
    stn_anomaly, x='ALTITUDE', y='temp_anomaly',
    color='year', symbol='year', trendline='ols',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'ALTITUDE':'Station Altitude (m)', 'temp_anomaly':'Temperature Anomaly (°C)'},
    title='Altitude vs Heat Anomaly — do elevated stations run cooler?',
    template='plotly_white', height=380
)
fig_alt_anom.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4)
fig_alt_anom.show()

---
## 5. Phase 4 — Sentinel-2 Feature Engineering

### 5a. Scene Inventory

| Scene date | Year | Notes |
|---|---|---|
| 2023-08-10 | 2023 | Early August — good coverage |
| 2023-08-20 | 2023 | **Representative scene for 2023** |
| 2023-08-23 | 2023 | Late August |
| 2024-07-30 | 2024 | ⚠️ Only 1 scene for 2024 — known limitation |
| 2025-07-02 | 2025 | Early summer |
| 2025-08-11 | 2025 | **Representative scene for 2025** |
| 2025-08-12 | 2025 | Consecutive-day pair — atmospheric check |

Bands: **B04** (Red) · **B08** (NIR) · **B11** (SWIR) — B12 excluded

In [ ]:
# ── Visual 12: Scene acquisition timeline ─────────────────────────────────────
scene_inv = pd.DataFrame({
    'scene_date': SCENE_DATES,
    'year': [d.year for d in SCENE_DATES],
    'label': [d.strftime('%b %d') for d in SCENE_DATES]
})
year_colors_map = {2023: '#4C72B0', 2024: '#DD8452', 2025: '#55A868'}

fig_tl = go.Figure()
for yr in YEARS:
    s = scene_inv[scene_inv['year'] == yr]
    fig_tl.add_trace(go.Scatter(
        x=s['scene_date'], y=[yr]*len(s),
        mode='markers+text',
        marker=dict(size=18, color=year_colors_map[yr], symbol='diamond'),
        text=s['label'], textposition='top center', name=str(yr)
    ))

# Annotate the 2024 limitation
fig_tl.add_annotation(
    x='2024-07-30', y=2024,
    text='⚠️ Only 1 scene<br>2024 limitation',
    showarrow=True, arrowhead=2, ax=60, ay=-50,
    font=dict(color='#DD8452', size=11)
)

fig_tl.update_layout(
    title='Sentinel-2 Acquisition Timeline — 7 Low-Cloud Summer Scenes',
    xaxis_title='Date',
    yaxis=dict(tickvals=YEARS, ticktext=[str(y) for y in YEARS]),
    height=320, template='plotly_white'
)
fig_tl.show()

In [ ]:
def sample_band_at_coords(band_path: Path, lats, lons) -> np.ndarray:
    """Sample a JP2 band at (lat,lon) coordinates."""
    with rasterio.open(band_path) as src:
        from pyproj import Transformer
        tr = Transformer.from_crs('EPSG:4326', src.crs.to_epsg(), always_xy=True)
        xs, ys = tr.transform(lons, lats)
        rows, cols = rasterio.transform.rowcol(src.transform, xs, ys)
        rows = np.clip(rows, 0, src.height - 1)
        cols = np.clip(cols, 0, src.width  - 1)
        data = src.read(1)
        return data[rows, cols].astype(np.float32) / 10000.0


def load_scene_bands(scene_dir: Path, lats, lons):
    out = {}
    for band in ['B04', 'B08', 'B11']:
        jp2 = scene_dir / f'{band}.jp2'
        out[band] = sample_band_at_coords(jp2, lats, lons) if jp2.exists() \
                    else np.full(len(lats), np.nan)
    return out


def build_sentinel_features(stations_df: pd.DataFrame) -> pd.DataFrame:
    records = []
    lats, lons, ids = (stations_df['LATITUDE'].values,
                       stations_df['LONGITUDE'].values,
                       stations_df['ID'].values)
    eps = 1e-6
    known = set(d.strftime('%Y-%m-%d') for d in SCENE_DATES)
    scene_dirs = sorted(d for d in SENTINEL_DIR.glob('????-??-??') if d.name in known)

    for scene_dir in scene_dirs:
        sd = pd.Timestamp(scene_dir.name)
        print(f'  Processing {scene_dir.name}')
        b = load_scene_bands(scene_dir, lats, lons)
        ndvi = (b['B08'] - b['B04']) / (b['B08'] + b['B04'] + eps)
        ndbi = (b['B11'] - b['B08']) / (b['B11'] + b['B08'] + eps)
        for i, sid in enumerate(ids):
            records.append({'ID': sid, 'scene_date': sd,
                            'B04': b['B04'][i], 'B08': b['B08'][i], 'B11': b['B11'][i],
                            'NDVI': ndvi[i], 'NDBI': ndbi[i]})
    return pd.DataFrame(records)


sentinel_df = build_sentinel_features(stations)
sentinel_df['scene_label'] = sentinel_df['scene_date'].dt.strftime('%Y-%m-%d')
print(f'Sentinel features: {len(sentinel_df):,} rows')
sentinel_df.head(3)

In [ ]:
# ── Visual 13: NDVI / NDBI distributions ──────────────────────────────────────
fig_idx = make_subplots(rows=1, cols=2,
                        subplot_titles=['NDVI (vegetation)', 'NDBI (built-up)'])
fig_idx.add_trace(go.Histogram(x=sentinel_df['NDVI'].dropna(),
                               marker_color='#55A868', nbinsx=40), row=1, col=1)
fig_idx.add_trace(go.Histogram(x=sentinel_df['NDBI'].dropna(),
                               marker_color='#C44E52', nbinsx=40), row=1, col=2)
fig_idx.update_layout(
    title='Spectral Index Distributions — all stations x 7 scenes',
    template='plotly_white', height=360, showlegend=False
)
fig_idx.show()

In [ ]:
# ── Visual 14: NDVI by scene — vegetation greenness varies across dates ────────
fig_ndvi_sc = px.box(
    sentinel_df, x='scene_label', y='NDVI', color='scene_label',
    title='NDVI per Scene — greenness across 7 acquisition dates',
    labels={'scene_label': 'Scene', 'NDVI': 'NDVI'},
    template='plotly_white', height=380
)
fig_ndvi_sc.update_layout(showlegend=False)
fig_ndvi_sc.show()

### 5b. Temporal Alignment & Final Merge

In [ ]:
def nearest_scene(d):
    idx = np.argmin(np.abs((SCENE_DATES - d).total_seconds()))
    return SCENE_DATES[idx]

sensor_daily['scene_date'] = sensor_daily['date'].apply(nearest_scene)

merged = sensor_daily.merge(sentinel_df, on=['ID', 'scene_date'], how='inner')
print(f'Merged rows: {len(merged):,}  (year breakdown: {merged["year"].value_counts().to_dict()})')
merged[['NDVI','NDBI','temp_anomaly']].describe().round(3)

In [ ]:
# ── Visual 15: NDVI vs Heat Anomaly — the green-cooling effect ────────────────
fig_nv = px.scatter(
    merged, x='NDVI', y='temp_anomaly', color='year',
    trendline='ols',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'NDVI': 'NDVI (vegetation density)',
            'temp_anomaly': 'Heat Anomaly (°C vs city mean)'},
    title='NDVI vs Heat Anomaly — parks and trees drive negative anomaly',
    template='plotly_white', height=420
)
fig_nv.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4)
fig_nv.show()

In [ ]:
# ── Visual 16: NDBI vs Heat Anomaly — the urban surface heating effect ─────────
fig_nb = px.scatter(
    merged, x='NDBI', y='temp_anomaly', color='year',
    trendline='ols',
    color_discrete_sequence=px.colors.qualitative.Set2,
    labels={'NDBI': 'NDBI (built-up intensity)',
            'temp_anomaly': 'Heat Anomaly (°C vs city mean)'},
    title='NDBI vs Heat Anomaly — pavements and rooftops drive positive anomaly',
    template='plotly_white', height=420
)
fig_nb.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.4)
fig_nb.show()

---
## 6. Phase 5 — ML Modeling

**Target:** `temp_anomaly` — °C above/below city daily mean  
**Features:** NDVI, NDBI, HUMIDITY, ALTITUDE, wind_ms, SOLARRADIATION

| Feature | Source | What it captures |
|---|---|---|
| NDVI | Sentinel-2 | Parks, trees, water → cooling |
| NDBI | Sentinel-2 | Roads, rooftops, concrete → heating |
| HUMIDITY | Leuven.cool | Evapotranspiration proxy |
| ALTITUDE | STATIONS.csv | Elevation cooling effect |
| wind_ms | Leuven.cool | Ventilation / heat dispersal |
| SOLARRADIATION | Leuven.cool | Direct solar load |

In [ ]:
FEATURES = ['NDVI', 'NDBI', 'HUMIDITY', 'ALTITUDE', 'wind_ms', 'SOLARRADIATION']
TARGET   = 'temp_anomaly'

ml_df = merged[FEATURES + [TARGET, 'year']].dropna()

scaler      = StandardScaler()
X           = scaler.fit_transform(ml_df[FEATURES])
y           = ml_df[TARGET].values
year_labels = ml_df['year'].values

print(f'ML dataset: {len(ml_df):,} rows')
print(f'Target range: {y.min():.2f} to {y.max():.2f} C')
print(f'Target std  : {y.std():.3f} C')

### 6a. Train / Test Split — 80/20 + 5-Fold Cross-Validation

In [ ]:
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
for train_idx, test_idx in sss.split(X, year_labels):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    years_test      = year_labels[test_idx]

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'5-Fold CV → ~{len(X_train)//N_FOLDS:,} rows per fold')

### 6b. Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=12, min_samples_leaf=2,
    n_jobs=-1, random_state=RANDOM_STATE
)
rf_cv = np.sqrt(-cross_val_score(
    rf, X_train, y_train, cv=kf,
    scoring='neg_mean_squared_error', n_jobs=-1
))
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)
print(f'RF  RMSE: {rf_rmse:.3f} | MAE: {rf_mae:.3f} | R2: {rf_r2:.3f}')
print(f'RF  5-CV: {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}')

### 6c. XGBoost

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    subsample=0.8, tree_method='hist',
    random_state=RANDOM_STATE, verbosity=0
)
xgb_cv = np.sqrt(-cross_val_score(
    xgb_model, X_train, y_train, cv=kf,
    scoring='neg_mean_squared_error', n_jobs=-1
))
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_r2   = r2_score(y_test, xgb_pred)
print(f'XGB RMSE: {xgb_rmse:.3f} | MAE: {xgb_mae:.3f} | R2: {xgb_r2:.3f}')
print(f'XGB 5-CV: {xgb_cv.mean():.3f} +/- {xgb_cv.std():.3f}')

### 6d. CNN (Convolutional Neural Network)

Sentinel-2 bands (B04, B08, B11) extracted as **16×16 pixel patches** centred on each station.  
The CNN learns spatial context — green corridors, rooftop clusters, shadow patterns.  
Target is also `temp_anomaly`.

**Architecture:** `Conv2d → ReLU → MaxPool → Conv2d → ReLU → MaxPool → Flatten → Linear(64) → ReLU → Linear(1)`  
**Image patches · ReLU · Adam optimizer**

In [ ]:
class HeatCNN(nn.Module):
    """CNN for urban heat anomaly regression from Sentinel-2 image patches."""
    def __init__(self, in_channels=3, patch_size=PATCH_SIZE):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),
        )
        flat = 32 * (patch_size // 4) ** 2
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 64), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.head(self.conv(x)).squeeze(1)


def extract_patch(band_path: Path, lat, lon, size=PATCH_SIZE):
    with rasterio.open(band_path) as src:
        from pyproj import Transformer
        tr = Transformer.from_crs('EPSG:4326', src.crs.to_epsg(), always_xy=True)
        x, y = tr.transform(lon, lat)
        r, c = rasterio.transform.rowcol(src.transform, x, y)
        half = size // 2
        window = rasterio.windows.Window(max(0,c-half), max(0,r-half), size, size)
        patch  = src.read(1, window=window).astype(np.float32) / 10000.0
        if patch.shape != (size, size):
            pad = np.zeros((size, size), dtype=np.float32)
            pad[:patch.shape[0], :patch.shape[1]] = patch
            patch = pad
    return patch


def build_patch_dataset(merged_df, patch_size=PATCH_SIZE):
    patches, targets = [], []
    for _, row in merged_df.iterrows():
        scene_dir = SENTINEL_DIR / str(row['scene_date'].date())
        bands, ok = [], True
        for band in ['B04','B08','B11']:
            jp2 = scene_dir / f'{band}.jp2'
            if not jp2.exists(): ok = False; break
            bands.append(extract_patch(jp2, row['LATITUDE'], row['LONGITUDE'], patch_size))
        if ok:
            patches.append(np.stack(bands))
            targets.append(row[TARGET])
    return np.array(patches, dtype=np.float32), np.array(targets, dtype=np.float32)


X_patches, y_patches = build_patch_dataset(merged)
print(f'Patch dataset: {X_patches.shape}')

In [ ]:
# ── Visual 17: Sample patches — colour encodes heat anomaly ───────────────────
# Sort patches by anomaly so we see coolest vs hottest
sort_idx   = np.argsort(y_patches)
cool_idx   = sort_idx[:3]    # coldest 3
hot_idx    = sort_idx[-3:]   # hottest 3
show_idx   = list(cool_idx) + list(hot_idx)
show_labels = [f'Cool {y_patches[i]:.1f}°C' for i in cool_idx] + \
              [f'Hot +{y_patches[i]:.1f}°C' for i in hot_idx]

band_names = ['B04 (Red)', 'B08 (NIR)', 'B11 (SWIR)']
fig, axes  = plt.subplots(6, 3, figsize=(9, 18))
for row_i, (patch_i, lbl) in enumerate(zip(show_idx, show_labels)):
    for col_j in range(3):
        ax = axes[row_i, col_j]
        ax.imshow(X_patches[patch_i, col_j], cmap='viridis', vmin=0, vmax=0.4)
        if row_i == 0: ax.set_title(band_names[col_j], fontsize=10)
        if col_j == 0: ax.set_ylabel(lbl, fontsize=9, rotation=0, labelpad=60, va='center')
        ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Coolest vs Hottest Station Patches (16×16 px) — anomaly target',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
Xp_tr, Xp_te, yp_tr, yp_te = tts(
    X_patches, y_patches, test_size=0.2, random_state=RANDOM_STATE
)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(Xp_tr), torch.from_numpy(yp_tr)),
    batch_size=32, shuffle=True
)

cnn_model   = HeatCNN().to(DEVICE)
optimizer   = optim.Adam(cnn_model.parameters(), lr=5e-4, weight_decay=1e-4)
criterion   = nn.MSELoss()
EPOCHS      = 20   # raise to 50 for full run
loss_hist   = []

cnn_model.train()
for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(cnn_model(xb), yb)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    avg = epoch_loss / len(train_loader)
    loss_hist.append(avg)
    if epoch % 4 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{EPOCHS}  loss: {avg:.4f}')

cnn_model.eval()
with torch.no_grad():
    cnn_pred = cnn_model(torch.from_numpy(Xp_te).to(DEVICE)).cpu().numpy()

cnn_rmse = np.sqrt(mean_squared_error(yp_te, cnn_pred))
cnn_mae  = mean_absolute_error(yp_te, cnn_pred)
cnn_r2   = r2_score(yp_te, cnn_pred)
print(f'CNN  RMSE: {cnn_rmse:.3f} | MAE: {cnn_mae:.3f} | R2: {cnn_r2:.3f}')

In [ ]:
# ── Visual 18: CNN training loss curve ───────────────────────────────────────
fig_loss = px.line(
    x=list(range(1, EPOCHS+1)), y=loss_hist,
    markers=True,
    labels={'x':'Epoch', 'y':'MSE Loss'},
    title='CNN Training Loss — anomaly target',
    template='plotly_white', height=350
)
fig_loss.update_traces(line_color='#C44E52')
fig_loss.show()

### 6e. Hyperparameter Tuning — XGBoost

In [ ]:
search = RandomizedSearchCV(
    xgb.XGBRegressor(tree_method='hist', verbosity=0, random_state=RANDOM_STATE),
    param_distributions={
        'n_estimators':  [100, 200, 300],
        'max_depth':     [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'subsample':     [0.7, 0.8, 1.0],
    },
    n_iter=12, scoring='neg_root_mean_squared_error',
    cv=kf, n_jobs=-1, random_state=RANDOM_STATE, verbose=1
)
search.fit(X_train, y_train)
best_xgb  = search.best_estimator_
best_pred = best_xgb.predict(X_test)
best_rmse = np.sqrt(mean_squared_error(y_test, best_pred))
best_mae  = mean_absolute_error(y_test, best_pred)
best_r2   = r2_score(y_test, best_pred)
print(f'Best params: {search.best_params_}')
print(f'Tuned XGB  RMSE: {best_rmse:.3f} | MAE: {best_mae:.3f} | R2: {best_r2:.3f}')

---
## 7. Phase 6 — Evaluation & Model Comparison

### 7a. Metrics Summary

In [ ]:
results = pd.DataFrame([
    {'Model':'Random Forest',   'RMSE':rf_rmse,   'MAE':rf_mae,   'R2':rf_r2,   'CV RMSE':rf_cv.mean()},
    {'Model':'XGBoost',         'RMSE':xgb_rmse,  'MAE':xgb_mae,  'R2':xgb_r2,  'CV RMSE':xgb_cv.mean()},
    {'Model':'XGBoost (tuned)', 'RMSE':best_rmse, 'MAE':best_mae, 'R2':best_r2, 'CV RMSE':abs(search.best_score_)},
    {'Model':'CNN',             'RMSE':cnn_rmse,  'MAE':cnn_mae,  'R2':cnn_r2,  'CV RMSE':float('nan')},
]).set_index('Model').round(4)
print(results.to_string())

In [ ]:
# ── Visual 19: RMSE & MAE comparison ──────────────────────────────────────────
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
models = results.index.tolist()

fig_cmp = make_subplots(rows=1, cols=2,
                        subplot_titles=['RMSE (°C anomaly)', 'MAE (°C anomaly)'])
for col_i, metric in enumerate(['RMSE','MAE'], start=1):
    fig_cmp.add_trace(go.Bar(
        x=models, y=results[metric].values,
        marker_color=colors, showlegend=False,
        text=results[metric].round(3).values, textposition='outside'
    ), row=1, col=col_i)
fig_cmp.update_layout(title='Model Comparison — Heat Anomaly Prediction',
                      height=420, template='plotly_white')
fig_cmp.show()

In [ ]:
# ── Visual 20: Radar chart ────────────────────────────────────────────────────
radar_df = results[['RMSE','MAE','R2']].copy()
eps = 1e-6
radar_df['RMSE_score'] = 1 - (radar_df['RMSE'] - radar_df['RMSE'].min()) / (radar_df['RMSE'].max() - radar_df['RMSE'].min() + eps)
radar_df['MAE_score']  = 1 - (radar_df['MAE']  - radar_df['MAE'].min())  / (radar_df['MAE'].max()  - radar_df['MAE'].min()  + eps)
radar_df['R2_score']   = (radar_df['R2'] - radar_df['R2'].min()) / (radar_df['R2'].max() - radar_df['R2'].min() + eps)

cats = ['RMSE score','MAE score','R2 score']
fig_radar = go.Figure()
for i, (model, row) in enumerate(radar_df.iterrows()):
    vals = [row['RMSE_score'], row['MAE_score'], row['R2_score']]
    fig_radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=cats + [cats[0]],
        fill='toself', name=model,
        line_color=colors[i], opacity=0.6
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    title='Model Performance Radar (normalised — higher = better)',
    template='plotly_white', height=420
)
fig_radar.show()

In [ ]:
# ── Visual 21: RMSE per year ──────────────────────────────────────────────────
year_records = []
for yr in YEARS:
    mask = years_test == yr
    if mask.sum() == 0: continue
    for mname, preds in [('Random Forest', rf_pred), ('XGBoost (tuned)', best_pred)]:
        year_records.append({
            'Year': yr, 'Model': mname,
            'RMSE': np.sqrt(mean_squared_error(y_test[mask], preds[mask])),
            'MAE':  mean_absolute_error(y_test[mask], preds[mask])
        })
year_df = pd.DataFrame(year_records)

fig_yr = px.bar(
    year_df, x='Year', y='RMSE', color='Model', barmode='group',
    title='RMSE per Year — note 2024 is worse (only 1 Sentinel-2 scene)',
    color_discrete_sequence=['#4C72B0','#55A868'],
    template='plotly_white', height=380
)
fig_yr.show()
print(year_df.round(4).to_string(index=False))

In [ ]:
# ── Visual 22: Predicted vs Actual anomaly scatter ────────────────────────────
pva = pd.DataFrame({'Actual': y_test, 'RF': rf_pred, 'XGB': best_pred})

fig_pva = make_subplots(rows=1, cols=2,
                        subplot_titles=['Random Forest', 'XGBoost (tuned)'])
for col_i, (m, col) in enumerate([('RF','#4C72B0'),('XGB','#55A868')], start=1):
    fig_pva.add_trace(go.Scatter(
        x=pva['Actual'], y=pva[m], mode='markers',
        marker=dict(color=col, opacity=0.6, size=6), showlegend=False
    ), row=1, col=col_i)
    lim = [pva['Actual'].min(), pva['Actual'].max()]
    fig_pva.add_trace(go.Scatter(x=lim, y=lim, mode='lines',
        line=dict(color='red', dash='dash'), showlegend=False), row=1, col=col_i)

fig_pva.update_xaxes(title_text='Actual anomaly (°C)')
fig_pva.update_yaxes(title_text='Predicted anomaly (°C)')
fig_pva.update_layout(
    title='Predicted vs Actual Heat Anomaly',
    height=420, template='plotly_white'
)
fig_pva.show()

In [ ]:
# ── Visual 23: Feature importance ────────────────────────────────────────────
imp_rf  = pd.Series(rf.feature_importances_,       index=FEATURES)
imp_xgb = pd.Series(best_xgb.feature_importances_, index=FEATURES)

fig_imp = make_subplots(rows=1, cols=2,
                        subplot_titles=['Random Forest', 'XGBoost (tuned)'])
for col_i, (imp, col) in enumerate(
        [(imp_rf.sort_values(),'#4C72B0'), (imp_xgb.sort_values(),'#55A868')],
        start=1):
    fig_imp.add_trace(go.Bar(
        x=imp.values, y=imp.index,
        orientation='h', marker_color=col, showlegend=False
    ), row=1, col=col_i)
fig_imp.update_layout(title='Feature Importance — what drives heat anomaly?',
                      height=380, template='plotly_white')
fig_imp.show()

In [ ]:
tabular = results.drop('CNN')
best_name = tabular['RMSE'].idxmin()
best_model_obj = {'Random Forest': rf,
                  'XGBoost': xgb_model,
                  'XGBoost (tuned)': best_xgb}[best_name]

print(f'Best model  : {best_name}')
print(f'Test RMSE   : {results.loc[best_name, "RMSE"]:.4f} C anomaly')
print('Selected as spatial predictor.')

---
## 8. Phase 7 — Heat Risk Maps

The model now predicts **temperature anomaly** (°C vs city mean) over a dense grid.  
Blue = cooler than city average (parks, trees, water).  
Red  = hotter than city average (pavements, rooftops, dense urban fabric).

In [ ]:
GRID_STEPS = 60

lat_grid  = np.linspace(LAT_MIN, LAT_MAX, GRID_STEPS)
lon_grid  = np.linspace(LON_MIN, LON_MAX, GRID_STEPS)
LAT_G, LON_G = np.meshgrid(lat_grid, lon_grid)
n_grid    = LAT_G.size
grid_lats = LAT_G.ravel()
grid_lons = LON_G.ravel()

def make_grid_X(scene_name, year):
    """Build feature matrix for the prediction grid using one scene."""
    scene_dir  = SENTINEL_DIR / scene_name
    bands_grid = {}
    eps        = 1e-6
    for band in ['B04','B08','B11']:
        jp2 = scene_dir / f'{band}.jp2'
        bands_grid[band] = sample_band_at_coords(jp2, grid_lats, grid_lons) \
                           if jp2.exists() else np.full(n_grid, np.nan)

    ndvi = (bands_grid['B08'] - bands_grid['B04']) / (bands_grid['B08'] + bands_grid['B04'] + eps)
    ndbi = (bands_grid['B11'] - bands_grid['B08']) / (bands_grid['B11'] + bands_grid['B08'] + eps)

    yr_sensor = df[df['year'] == year]
    gX = np.column_stack([
        ndvi, ndbi,
        np.full(n_grid, yr_sensor['HUMIDITY'].median()),
        np.full(n_grid, yr_sensor['ALTITUDE'].median()),
        np.full(n_grid, yr_sensor['wind_ms'].median()),
        np.full(n_grid, yr_sensor['SOLARRADIATION'].median()),
    ])
    nan_mask = np.any(np.isnan(gX), axis=1)
    for c in range(gX.shape[1]):
        gX[nan_mask, c] = np.nanmedian(gX[:, c])
    return gX


# Latest scene — overall map
latest_scene_name = REPR_SCENES[2025]
gX_latest = make_grid_X(latest_scene_name, 2025)
grid_anomaly = best_model_obj.predict(scaler.transform(gX_latest)).reshape(LAT_G.shape)

print(f'Grid anomaly range: {grid_anomaly.min():.2f} to {grid_anomaly.max():.2f} C')

In [ ]:
# ── Visual 24: Interactive Urban Heat Anomaly Dashboard ───────────────────────
anom_abs = np.abs(grid_anomaly).max()

fig_map = go.Figure(go.Heatmap(
    z=grid_anomaly, x=lon_grid, y=lat_grid,
    colorscale='RdBu_r',
    zmid=0,
    zmin=-anom_abs, zmax=anom_abs,
    colorbar=dict(title='Anomaly (°C)'),
    hovertemplate='Lon: %{x:.4f}<br>Lat: %{y:.4f}<br>Anomaly: %{z:.2f}°C<extra></extra>'
))
fig_map.add_trace(go.Scatter(
    x=stations['LONGITUDE'], y=stations['LATITUDE'],
    mode='markers',
    marker=dict(color='black', size=5, symbol='circle'),
    name='Stations',
    text=stations['ID'],
    hovertemplate='%{text}<extra></extra>'
))
fig_map.update_layout(
    title='Urban Heat Anomaly · Leuven — Blue=Cool (parks/trees), Red=Hot (pavements/rooftops)',
    xaxis_title='Longitude', yaxis_title='Latitude',
    height=580, template='plotly_white'
)
fig_map.show()

In [ ]:
# ── Visual 25: Multi-year anomaly comparison ──────────────────────────────────
year_grids = {}
for yr in YEARS:
    gX_yr = make_grid_X(REPR_SCENES[yr], yr)
    year_grids[yr] = best_model_obj.predict(
        scaler.transform(gX_yr)
    ).reshape(LAT_G.shape)

fig_multi = make_subplots(
    rows=1, cols=3,
    subplot_titles=[f'{yr}  ({REPR_SCENES[yr]})' for yr in YEARS],
    horizontal_spacing=0.05
)
sym_max = max(np.abs(g).max() for g in year_grids.values())

for col_i, yr in enumerate(YEARS, start=1):
    fig_multi.add_trace(go.Heatmap(
        z=year_grids[yr], x=lon_grid, y=lat_grid,
        colorscale='RdBu_r', zmid=0,
        zmin=-sym_max, zmax=sym_max,
        showscale=(col_i == 3),
        colorbar=dict(title='Anomaly °C', x=1.01)
    ), row=1, col=col_i)

fig_multi.update_layout(
    title='Multi-Year Urban Heat Anomaly — same scale, same model',
    height=430, template='plotly_white'
)
fig_multi.show()

In [ ]:
# ── Visual 26: Heat risk category map (anomaly-based) ─────────────────────────
anom_flat = grid_anomaly.ravel()
risk_cat  = pd.cut(
    anom_flat,
    bins=[-np.inf, -2, -0.5, 0.5, 2, np.inf],
    labels=['Very Cool','Cool','Neutral','Warm','Hot Spot']
)
risk_df = pd.DataFrame({
    'lat': grid_lats, 'lon': grid_lons,
    'anomaly': anom_flat, 'risk': risk_cat
})

fig_risk = px.scatter_mapbox(
    risk_df, lat='lat', lon='lon',
    color='risk',
    color_discrete_map={
        'Very Cool': '#313695',
        'Cool':      '#74ADD1',
        'Neutral':   '#FFFFBF',
        'Warm':      '#F46D43',
        'Hot Spot':  '#A50026'
    },
    hover_data={'anomaly': ':.2f'},
    zoom=12, mapbox_style='carto-positron',
    title='Urban Heat Risk Categories · Leuven (anomaly-based)',
    height=560
)
fig_risk.show()

---
## 9. Phase 8 — Belgian Urban Heat Dashboard Vision

### 9a. City-level Spike Detection

Using RMI AWS data, we can detect **which years had anomalous heat spikes** across Belgian cities.  
This is the foundation of the web dashboard: "Was 2025 hotter than 2023 in Brussels?"

In [ ]:
# ── Visual 27: City × Year heat anomaly heatmap ───────────────────────────────
rmi_q3_copy = rmi_daily[
    rmi_daily['date'].dt.month.isin([7,8,9])
].copy()
rmi_q3_copy['year'] = rmi_q3_copy['date'].dt.year

pivot = rmi_q3_copy.groupby(['year','city'])['value'].mean().unstack('city')
baseline_row = pivot.loc[2023]
anomaly_pivot = pivot.subtract(baseline_row, axis=1)

fig, ax = plt.subplots(figsize=(8, 3))
sns.heatmap(
    anomaly_pivot.T.round(2), annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax, cbar_kws={'label': '°C vs 2023 baseline'}
)
ax.set_title('City × Year Heat Anomaly — vs 2023 baseline (RMI AWS)',
             fontsize=12, pad=10)
ax.set_xlabel('Year'); ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# ── Visual 28: Rolling 7-day mean — heat wave detection ───────────────────────
rmi_leuven = rmi_daily[
    (rmi_daily['city'] == 'Leuven (Diest)') &
    (rmi_daily['date'].dt.month.isin([7,8,9]))
].copy().sort_values('date')

rmi_leuven['year'] = rmi_leuven['date'].dt.year
rmi_leuven['roll7'] = rmi_leuven['value'].rolling(7, min_periods=1).mean()

fig_roll = px.line(
    rmi_leuven, x='date', y='roll7', color='year',
    color_discrete_sequence=['#4C72B0','#DD8452','#55A868'],
    labels={'roll7': '7-day rolling mean temp (°C)', 'date': 'Date'},
    title='Leuven 7-Day Rolling Temperature — Heat Wave Periods Visible',
    template='plotly_white', height=380
)
# Mark scene dates
for sd in SCENE_DATES:
    fig_roll.add_vline(x=sd, line_dash='dot', line_color='gray',
                       opacity=0.5, annotation_text=sd.strftime('%m/%d'),
                       annotation_position='top')
fig_roll.show()

### 9b. Website Architecture — Belgian Urban Heat Monitor

```
┌─────────────────────────────────────────────────────────────────────┐
│                  Belgian Urban Heat Monitor                          │
│            urban-heat-belgium.be  (proposed)                        │
├─────────────────────────────────────────────────────────────────────┤
│  HEADER: City selector  |  Year slider  |  Variable toggle          │
│          [Leuven ▼]     |  [2023–2025]  |  [Anomaly / Absolute]     │
├──────────────────────────────┬──────────────────────────────────────┤
│  MAP PANEL                   │  METRICS PANEL                       │
│  Plotly mapbox heatmap        │  • City mean anomaly vs baseline     │
│  Blue=Cool / Red=Hot          │  • Year-over-year spike: +X°C        │
│  Station dots overlay         │  • Hotspot count this year           │
│                              │  • Best model RMSE badge             │
├──────────────────────────────┴──────────────────────────────────────┤
│  COMPARISON ROW:  2023  │  2024  │  2025  (same scale, same model)  │
├─────────────────────────────────────────────────────────────────────┤
│  TIMESERIES: 7-day rolling temp per city  |  RMI vs Leuven.cool     │
└─────────────────────────────────────────────────────────────────────┘
```

**Tech stack recommendation:**

| Layer | Tool | Why |
|---|---|---|
| Backend | Python + FastAPI | Serve model predictions as JSON API |
| Maps | Plotly Dash or Streamlit | Fastest path to interactive web maps |
| Data | openkmi + Leuven.cool | Auto-refresh on new RMI data |
| Hosting | Render.com / Hugging Face Spaces | Free tier, no infrastructure needed |

**Data pipeline for the website:**
```
RMI AWS (daily pull via openkmi)  ─┐
Leuven.cool (Q3 CSV per year)      ├─► anomaly_engine.py ─► predictions ─► dashboard
Sentinel-2 (new scenes via API)   ─┘
```

---
## 10. Phase 9 — Deployment

In [ ]:
fig_map.write_html('belgian_heat_dashboard.html')
fig_risk.write_html('leuven_heat_risk_categories.html')
fig_multi.write_html('leuven_heat_multiyear.html')

print('Exported:')
print('  belgian_heat_dashboard.html   — interactive anomaly heatmap')
print('  leuven_heat_risk_categories.html — risk category map')
print('  leuven_heat_multiyear.html     — 2023 / 2024 / 2025 comparison')

---
### Key changes in this version

| Change | What it fixes |
|---|---|
| **Target = `temp_anomaly`** | Model now learns spatial heat patterns, not regional weather |
| **Scene-windowed loading** | Sensor data matched to actual scene dates, far more merged rows |
| **RMI AWS integration** | Official Belgium baseline, enables multi-city comparison |
| **2024 limitation documented** | Transparent about 1-scene problem, excluded from maps |
| **Maps use symmetric RdBu_r** | Blue=cool parks, Red=hot pavements — meaningful colours |
| **Anomaly-based risk categories** | `Very Cool / Cool / Neutral / Warm / Hot Spot` |
| **Website architecture** | Dashboard design + tech stack for Belgian Urban Heat Monitor |

---
*Belgian Urban Heat Monitor · Leuven 2023–2025 — AI-assisted urban heat anomaly detection using satellite imagery and environmental sensing.*

**Data sources:**  
Leuven.cool citizen science network — [Zenodo DOI 10.5281/zenodo.14893734](https://zenodo.org/records/14893734)  
RMI Automatic Weather Stations — [opendata.meteo.be](https://opendata.meteo.be)  
Sentinel-2 L2A imagery — ESA Copernicus · 7 low-cloud summer scenes 2023–2025